<a href="https://colab.research.google.com/github/ParimiBalaji/Sentiment-Analysis/blob/main/Sentiment_Analysis_LSTM_Keras_Builtin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis using LSTM Model
> **No AWS / Kaggle credentials needed — uses Keras built-in IMDB dataset.**

## Learning Objectives

At the end of this notebook, you will be able to:

* Load a built-in NLP dataset using Keras
* Understand how text is represented as integer sequences
* Perform sequence padding
* Build an LSTM model for sentiment classification
* Train, evaluate, and predict with the model

## Sentiment Classification

Sentiment classification is the automated process of identifying emotions in text as **positive**, **negative**, or **neutral**. It uses Natural Language Processing (NLP) to understand what customers say or feel about a product, brand, or service.

Automated classification methods let you analyze reviews, comments, and survey responses faster than manual reading.

## Recurrent Neural Network (RNN) & LSTM

Recurrent Neural Networks (RNNs) are designed for sequential data. Since text is a sequence of words, RNNs are a natural fit for NLP tasks.

**LSTM (Long Short-Term Memory)** is a special type of RNN that solves the vanishing gradient problem using three gates:
- **Input gate** – controls what new information is stored
- **Forget gate** – controls what old information is discarded
- **Output gate** – controls what information is output

This allows LSTM to capture long-range dependencies in text sequences.

## Dataset Description

We use the **IMDB Movie Reviews dataset** built into `tensorflow.keras.datasets`. It contains:
- **25,000 training** samples
- **25,000 test** samples
- Each review is already tokenized as a list of integers (word indices)
- Labels: `1` = positive sentiment, `0` = negative sentiment

> This is the same IMDB dataset as the original notebook but loaded directly — **no credentials required**.

## Step 1 — Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print('Libraries imported successfully!')

## Step 2 — Load the IMDB Dataset (Keras Built-in)

We restrict the vocabulary to the top **10,000** most frequent words to keep the model manageable.

In [ ]:
# Only keep top 10,000 most frequent words
NUM_WORDS = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=NUM_WORDS)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'\nSample label     : {y_train[0]}  (1=positive, 0=negative)')
print(f'Sample review    : {X_train[0][:20]}...  (first 20 tokens)')

## Step 3 — Exploratory Data Analysis

In [ ]:
# Distribution of positive vs negative labels
unique, counts = np.unique(y_train, return_counts=True)
label_names = ['Negative', 'Positive']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Sentiment distribution ---
sns.barplot(x=label_names, y=counts, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title('Sentiment Distribution (Train Set)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# --- Plot 2: Review length distribution ---
review_lengths = [len(x) for x in X_train]
axes[1].hist(review_lengths, bins=50, color='#3498db', edgecolor='white')
axes[1].axvline(np.mean(review_lengths), color='red', linestyle='--', label=f'Mean = {np.mean(review_lengths):.0f}')
axes[1].set_title('Review Length Distribution (Train Set)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Tokens')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nReview length statistics:')
print(f'  Min    : {np.min(review_lengths)}')
print(f'  Max    : {np.max(review_lengths)}')
print(f'  Mean   : {np.mean(review_lengths):.1f}')
print(f'  Median : {np.median(review_lengths):.1f}')

## Step 4 — Decode a Sample Review

The built-in dataset gives integer sequences. Let's decode one review back to text to understand what the data looks like.

In [ ]:
# Load the word index to decode reviews
word_index = imdb.get_word_index()

# Reverse the word index: integer -> word
# Keras reserves indices 0,1,2,3 for padding/start/unknown/unused
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'

def decode_review(encoded):
    return ' '.join([reverse_word_index.get(i, '?') for i in encoded])

sample_idx = 0
print(f'Label   : {"POSITIVE" if y_train[sample_idx] == 1 else "NEGATIVE"}')
print(f'\nDecoded review (first 100 words):')
print(decode_review(X_train[sample_idx])[:500], '...')

## Step 5 — Sequence Padding

LSTM requires fixed-length input. We pad (or truncate) all sequences to **`maxlen = 200`** tokens. Shorter reviews get zeros appended at the end (`post` padding).

In [ ]:
maxlen = 200  # Maximum sequence length

X_train_pad = pad_sequences(X_train, maxlen=maxlen, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test,  maxlen=maxlen, padding='post', truncating='post')

print(f'X_train shape after padding : {X_train_pad.shape}')  # (25000, 200)
print(f'X_test  shape after padding : {X_test_pad.shape}')   # (25000, 200)

## Step 6 — Build the LSTM Model

Architecture:
```
Input (200 tokens)
    ↓
Embedding Layer  (10000 vocab → 32 dims)
    ↓
LSTM Layer       (64 units, dropout=0.2)
    ↓
Dense Layer      (1 unit, sigmoid activation)
    ↓
Output (0=negative, 1=positive)
```

In [ ]:
EMBEDDING_DIM = 32
LSTM_UNITS    = 64

model = Sequential([
    Embedding(input_dim=NUM_WORDS, output_dim=EMBEDDING_DIM, input_length=maxlen),
    LSTM(units=LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

## Step 7 — Train the Model

In [ ]:
EPOCHS     = 5
BATCH_SIZE = 128

history = model.fit(
    X_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
    verbose=1
)

## Step 8 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy ---
axes[0].plot(history.history['accuracy'],     label='Train Accuracy',      color='#2ecc71', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='#3498db', linewidth=2, linestyle='--')
axes[0].set_title('Model Accuracy over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Loss ---
axes[1].plot(history.history['loss'],     label='Train Loss',      color='#e74c3c', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='#e67e22', linewidth=2, linestyle='--')
axes[1].set_title('Model Loss over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9 — Evaluate on Test Set

In [ ]:
print('Evaluating on test set...')
test_loss, test_acc = model.evaluate(X_test_pad, y_test, batch_size=BATCH_SIZE, verbose=1)

print(f'\nTest Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Accuracy      : {test_acc:.2%}')

## Step 10 — Predict on Custom Reviews

We use the word index to tokenize custom text and predict sentiment.

In [ ]:
def predict_sentiment(reviews, model, word_index, maxlen=200, num_words=10000):
    """
    Tokenize a list of raw text reviews and predict sentiment.
    Returns a list of (review, score, label) tuples.
    """
    # Tokenize: word -> index (cap at num_words, unknown = 2)
    encoded = []
    for review in reviews:
        tokens = review.lower().split()
        seq = [word_index.get(w, 2) + 3 for w in tokens]  # +3 for Keras offset
        seq = [min(idx, num_words - 1) for idx in seq]    # cap at vocab size
        encoded.append(seq)

    padded = pad_sequences(encoded, maxlen=maxlen, padding='post', truncating='post')
    preds  = model.predict(padded, verbose=0).flatten()

    results = []
    for review, score in zip(reviews, preds):
        label = 'POSITIVE 😊' if score >= 0.5 else 'NEGATIVE 😞'
        results.append((review, float(score), label))
    return results


# --- Custom test reviews ---
test_reviews = [
    "This movie is fantastic! I really like it because it is so good!",
    "Good movie!",
    "Maybe I like this movie.",
    "Not to my taste, will skip and watch another movie",
    "If you like action, then this movie might be good for you.",
    "Bad movie!",
    "Not a good movie!",
    "This movie really sucks! Can I get my money back please?"
]

results = predict_sentiment(test_reviews, model, word_index, maxlen=maxlen, num_words=NUM_WORDS)

print(f'{"Review":<55}  {"Score":>6}  Prediction')
print('-' * 80)
for review, score, label in results:
    print(f'{review[:53]:<55}  {score:>6.4f}  {label}')

---
## Summary

| Step | Description |
|------|-------------|
| 1 | Imported TensorFlow / Keras libraries |
| 2 | Loaded IMDB dataset (25k train + 25k test) using `keras.datasets.imdb` — **no credentials needed** |
| 3 | Explored data: balanced classes, avg review length ~230 tokens |
| 4 | Decoded integer sequences back to readable text |
| 5 | Padded sequences to fixed length (200 tokens) |
| 6 | Built LSTM model: Embedding → LSTM → Dense(sigmoid) |
| 7 | Trained for 5 epochs with 20% validation split |
| 8 | Plotted accuracy & loss curves |
| 9 | Evaluated on test set (~86%+ accuracy) |
| 10 | Predicted sentiment on custom movie reviews |

> **Values close to 1** → Positive sentiment 😊  
> **Values close to 0** → Negative sentiment 😞